# 3. Treinamento em cinco milhões de ASTs RAW/TANH

O corpus sorteia cinco milhões de ASTs com até seis operações. Para cada AST, uma escolha com probabilidade 0,5 decide se `product_lags` será aplicado antes da canonização; a AST resultante é gravada em Parquet. O encoder recebe essa AST e o decoder prevê a forma ordenada correspondente, seja RAW ou protegida. O mesmo skeleton pertence a apenas uma partição; validação e teste retêm um exemplo por skeleton. Os modelos Transformer preveem (1) o skeleton e (2) o skeleton junto com coeficientes.

Os coeficientes-alvo pertencem à AST escolhida. Ao gerar séries pelo método Shao–Wu, o gerador acrescenta um fator de escala certificado. A proteção seletiva é uma construção do projeto inspirada nas condições de Shao e Wu (2007, Theorem 5.1).


In [ ]:
from pathlib import Path
import json
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').exists())
CONFIG_PATH=ROOT/'configs/canonicalization_5m_raw_tanh.json'
RUN_DIR=ROOT/'outputs/canonicalization/ast_5m_raw_tanh_20261101'
config=json.loads(CONFIG_PATH.read_text(encoding='utf8'))
print(config)


A configuração registra sementes, tamanho do corpus, separação disjunta por skeleton, deduplicação de validação e teste, arquitetura e métricas.

In [ ]:
import subprocess,sys,torch
device='cuda' if torch.cuda.is_available() else 'cpu'
command=[sys.executable,str(ROOT/'scripts/run_canonical_5m.py'),
         '--config',str(CONFIG_PATH.relative_to(ROOT)),
         '--run-dir',str(RUN_DIR.relative_to(ROOT)),
         '--device',device]
print(' '.join(command))
subprocess.run(command,cwd=ROOT,check=True)


A execução gera Parquet, verifica a separação dos skeletons, treina duas variantes e avalia a geração livre. Os artefatos ficam em `outputs/`.

In [ ]:
status=json.loads((RUN_DIR/'status.json').read_text(encoding='utf8'))
print(status['corpus'])
for name,result in status['results'].items():
    print(name,{key:result.get(key) for key in ('count','skeleton_exact','term_f1','joint_exact','best_epoch')})


O resumo mostra cardinalidade, diversidade estrutural e métricas de cada variante. O registro completo permanece neste notebook depois da execução.

## Registro do experimento

**Pergunta:** os Transformers aprendem a converter ASTs RAW e protegidas em skeletons e coeficientes para skeletons ausentes no treino? **Configuração:** `configs/canonicalization_5m_raw_tanh.json`, incluindo 5.000.000 ASTs, probabilidade de proteção 0,5, uma série, lags 1–12, 1–6 operações, grau máximo 4 e divisão 70/10/20. **Seeds:** 20261101, 20261102 e 20261103. **Proveniência:** `canonical_large_scale.py`, `canonical_data.py`, `canonical_models.py`, `shao_wu.py`, `canonical_polynomial.py`, configuração versionada e hashes registrados durante a execução. **Métricas:** skeleton exato, termo F1, acerto conjunto, tempo e contagens por partição. **Reprodução quando solicitada:** `.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace experiments/canonicalization/3_ast_encoder_decoder.ipynb`. Os resultados serão acrescentados após a execução.
